# Notebook 3
## Chronicling America 1906 Newspaper Corpus

provide an alternative cosine similarity search engine
- "home-made" vector-space cosine similarity system instead of PyTerrier

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

Why TfidfVectorizer
- its simple, hand-on, ready to use component
- algorithm light / built-in weight

In [3]:
DATA_DIR = Path("../data")
CORPUS = DATA_DIR / "corpus.parquet"

print("Current working directory:", Path.cwd())
print("Corpus path:", CORPUS)

Current working directory: /Users/alexhuynh/Documents/INFO376/376repo/NewsRank/scripts
Corpus path: ../data/corpus.parquet


In [4]:
corpus = pd.read_parquet(CORPUS)
corpus.head()

,doc_id,lccn,date,edition,page,raw_text,clean_text,token_count
0,sn83035387_1906_01_06_ed1_seq1,sn83035387,1906-01-06,1,1,TIIE G AZETTE\nTWENTY-THIRD YEAR. NO. 23.\nThe...,tiie azett twenti third year fashion uay autho...,3239
1,sn83035387_1906_01_06_ed1_seq2,sn83035387,1906-01-06,1,2,2\nTHE GAZETTE.\nPUBLISHED EVERY SATURDAY.\nSU...,gazett publish everi saturday subscript rate a...,3700
2,sn83035387_1906_01_06_ed1_seq3,sn83035387,1906-01-06,1,3,THE CATERERS' ASSOCIATION\nOF CLEVELAND\nWILL ...,cater associ cleveland give initi grand soire ...,2568
3,sn83035387_1906_01_06_ed1_seq4,sn83035387,1906-01-06,1,4,"4\nA Guaranteed Cure for Piles.\nItrhing, Pin....",guarante cure pile itrh pin bleed prtdrude pii...,2696
4,sn83035387_1906_01_13_ed1_seq1,sn83035387,1906-01-13,1,1,THE WT GAZETTE\nTWENTY-THIRD YEAR. NO. 24.\nTh...,gazett twenti third year fashion day gown make...,3260


In [5]:
corpus["clean_text"]

0      tiie azett twenti third year fashion uay autho...
1      gazett publish everi saturday subscript rate a...
2      cater associ cleveland give initi grand soire ...
3      guarante cure pile itrh pin bleed prtdrude pii...
4      gazett twenti third year fashion day gown make...
                             ...                        
603    loui palladium fttbl kvtcri batvrday aur tha p...
604    prof leon devoux born seer past master clairvo...
605    loui royal hous meet first friday night month ...
606    tri experi spent sioo vain search health miss ...
607    grand lodg unit brother friendship sister myst...
Name: clean_text, Length: 608, dtype: object

## Vectorization

In [6]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus["clean_text"])

In [7]:
num_docs, vocab_size = tfidf_matrix.shape
print("Corpus Statistics")
print("-----------------")
print(f"Number of documents: {num_docs}")
print(f"Vocabulary size: {vocab_size}")

Corpus Statistics
-----------------
Number of documents: 608
Vocabulary size: 55362


## Cosine Similarity 

### pilot case

In [8]:
query = "goods on sale"
query_vector = vectorizer.transform([query])
scores = cosine_similarity(query_vector, tfidf_matrix)
scores = scores.flatten()
ranked_indices = np.argsort(scores)[::-1]
ranked_indices[:10]
top_indices = ranked_indices[:10]

In [9]:
top_documents = corpus.iloc[top_indices]
top_documents = top_documents[["doc_id", "clean_text"]]
top_documents

,doc_id,clean_text
98,sn83035387_1906_06_23_ed1_seq3,localdepart notic subscrib subscrib receiv gaz...
511,sn84020235_1906_09_29_ed1_seq8,tailor fourteenth rfall winter suit winter sui...
354,sn84020235_1906_05_12_ed1_seq1,loui palladium vol xxii suprem master martin w...
518,sn84020235_1906_10_06_ed1_seq7,lumbago sciatica tmob mark jacob oil penetr sp...
212,sn84020235_1906_01_06_ed1_seq3,negro newspap th unit state tri hard get exact...
208,sn83035387_1906_12_29_ed1_seq3,localdepart notic subscrib subscrib receiv gaz...
367,sn84020235_1906_05_19_ed1_seq6,wast shadow found cure fifteen year buffer sto...
165,sn83035387_1906_10_13_ed1_seq4,nervou debil woman tell william pink pill made...
416,sn84020235_1906_06_30_ed1_seq7,missouri itg jarisdictioa grand lodg grand lod...
105,sn83035387_1906_07_07_ed1_seq2,gazett publish everi saturday subscript rate a...


## functions

In [10]:
def search_cosine(query, vectorizer, tfidf_matrix, corpus, top_k=10):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    ranked_indices = np.argsort(scores)[::-1]
    top_indices = ranked_indices[:top_k]
    
    results = corpus.iloc[top_indices].copy()
    results["score"] = scores[top_indices]
    
    return results[["doc_id", "score", "clean_text"]]

def get_cosine_scores(query, vectorizer, tfidf_matrix, corpus):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    score_df = corpus[["doc_id"]].copy()
    score_df["cosine_score"] = scores
    score_df = score_df.sort_values("cosine_score", ascending=False).reset_index(drop=True)
    score_df["cosine_rank"] = score_df.index + 1

    return score_df

def run_cosine_search(query, top_k=10, return_all_scores=False):
    if return_all_scores:
        return get_cosine_scores(query, vectorizer, tfidf_matrix, corpus)
    else:
        return search_cosine(query, vectorizer, tfidf_matrix, corpus, top_k=top_k)

In [11]:
run_cosine_search("roosevelt speech election", top_k=5)

,doc_id,score,clean_text
267,sn84020235_1906_02_24_ed1_seq2,0.124822,electa templ meet second thursday month knight...
207,sn83035387_1906_12_29_ed1_seq2,0.116967,gazett publish everi saturday subscript rate a...
580,sn84020235_1906_12_01_ed1_seq5,0.107030,prof leon deyoux born seer past master clairvo...
595,sn84020235_1906_12_15_ed1_seq4,0.090915,palladium fttbiubheid evept haturtm terad poat...
186,sn83035387_1906_11_24_ed1_seq1,0.083352,gazett twenti fourth year onio news sent mani ...


#### Export matrix / vectors

In [12]:
import pickle
import scipy.sparse as sp

sp.save_npz("../data/tfidf_matrix.npz", tfidf_matrix)
with open("../data/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)